# Camada Gold

# V-Credit | Camada Gold: `chamados_custos`

## O que este notebook faz

Lê as tabelas da Silver (`chamados`, `custos`, `canais` e `motivos`), cria uma chave substituta (`sk_chamado`) com SHA-256 e integra os dados em uma única tabela por meio de `left joins`. A tabela consolidada reúne informações dos chamados, custos, canais e motivos, servindo de base para as análises do case. Em seguida, executa consultas SQL para analisar chamados simples resolvidos pelo canal humano e chamados complexos resolvidos pelo autosserviço, estimando custos excedentes e economias potenciais.

- Entrada: `rocket_lab.vcredit_silver` (`chamados`, `custos`, `canais`, `motivos`).
- Saída: `rocket_lab.vcredit_gold.chamados_custos`.

## Como o notebook está organizado

| Bloco | Conteúdo |
|---|---|
| 0 | parâmetros de catálogo e schemas |
| 1 | leitura das tabelas da Silver |
| 2 | criação da chave substituta `sk_chamado` e seleção das colunas |
| 3 | integração dos chamados com custos, canais e motivos por `left joins` |
| 4 | gravação da tabela Gold em formato Delta e conferência dos dados |
| 5 | análise de chamados simples resolvidos pelo canal humano |
| 6 | análise de chamados complexos resolvidos pelo autosserviço |

## Definições usadas nas análises

Algumas regras foram adotadas para padronizar a construção da tabela e permitir a comparação de custos entre os canais.

Chave substituta
- `sk_chamado` é gerada a partir de `id_chamado` utilizando SHA-256. Para o mesmo ID de entrada, o hash é determinístico.
- O `id_chamado` original é mantido para relacionar os dados das diferentes tabelas.

Integração dos dados
- `chamados` é a tabela principal, enriquecida com informações de `custos`, `canais` e `motivos`.
- Os relacionamentos utilizam `left joins`, preservando os registros da tabela de chamados mesmo quando não existe correspondência nas tabelas relacionadas.
- A tabela final reúne informações de resolução, custo, tipo e status do canal, categoria e criticidade do motivo.

Custos e comparações
- Chamados simples são definidos como resolvidos e de baixa criticidade.
- Chamados complexos são definidos como resolvidos e de média ou alta criticidade.
- O custo excedente representa a diferença entre o custo real dos chamados simples atendidos pelo canal humano e o custo estimado caso fossem atendidos pelo autosserviço.
- O custo salvo representa a diferença entre o custo estimado dos chamados complexos no canal humano e o custo real desses chamados no autosserviço.
- As comparações utilizam o custo médio do canal de referência para estimar o custo alternativo. Portanto, os valores representam estimativas, não necessariamente economias efetivamente realizadas.

Persistência
- A tabela `chamados_custos` é gravada no formato Delta com `mode("overwrite")`, substituindo os dados existentes a cada execução.
- A opção `overwriteSchema` permite atualizar o schema da tabela durante a gravação.


## Bloco 0: definições

Os parâmetros abaixo definem os schemas de origem e destino utilizados na construção da camada Gold.

| Parâmetro | Valor | Significado |
|---|---|---|
| `catalog` | `rocket_lab` | catálogo utilizado no ambiente de dados |
| `silver_schema_name` | `vcredit_silver` | schema de origem, contendo as tabelas Silver |
| `silver_schema` | `rocket_lab.vcredit_silver` | caminho completo do schema de origem |
| `gold_schema_name` | `vcredit_gold` | schema de destino da camada Gold |
| `gold_schema` | `rocket_lab.vcredit_gold` | caminho completo do schema de destino |
| `sk_chamado` | SHA-256 de `id_chamado` | chave substituta determinística utilizada para identificar os chamados |


In [0]:
# Funções SQL do Spark (usadas como F.*) e suporte a funções de janela (Window)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Bloco 0 — catálogo e schemas de origem (Silver) e destino (Gold)

# Catálogo onde residem as camadas Silver e Gold
catalog = "rocket_lab"

# Schema de origem: tabelas Silver já tratadas
silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"  

# Schema de destino: tabela Gold consolidada
gold_schema_name = "vcredit_gold"
gold_schema = f"{catalog}.{gold_schema_name}" 

# Garante a existência do schema Gold antes da gravação (idempotente)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

### 1) Chamados Custos

In [0]:
# Bloco 1 — leitura das tabelas da Silver
chamados = spark.table(f"{silver_schema}.chamados")  
custos   = spark.table(f"{silver_schema}.custos")    
canais   = spark.table(f"{silver_schema}.canais")    
motivos  = spark.table(f"{silver_schema}.motivos")  

# Bloco 2 — chave substituta sk_chamado (SHA-256 de id_chamado) e seleção das colunas
dim_chamado = (
    chamados
    .withColumn("sk_chamado", F.sha2(F.col("id_chamado").cast("string"), 256))  
    .select("sk_chamado", "id_chamado", "motivo", "canal", "resolvido")
)

# Custo por chamado, ligado ao chamado por id_chamado
dim_custos = (
    custos
    .select("id_chamado", "custo")
)

# Atributos descritivos dos canais
dim_canais = (
    canais
    .select("nome_canal", "tipo_canal", "status_canal")
)

# Atributos descritivos dos motivos
dim_motivos = (
    motivos
    .select("nome_motivo", "categoria", "criticidade", "categoria_imputada")
)

# Bloco 3 — integração por left joins: preserva todos os chamados mesmo sem correspondência
chamados_custos = (
    dim_chamado
    .join(dim_custos, on="id_chamado", how="left")                                
    .join(dim_canais, dim_chamado.canal == dim_canais.nome_canal, how="left")       
    .join(dim_motivos, dim_chamado.motivo == dim_motivos.nome_motivo, how="left")   
    .select("sk_chamado", "motivo", "canal", "resolvido", "custo", "nome_canal", "tipo_canal", "status_canal", "nome_motivo", "categoria", "criticidade", "categoria_imputada")
)

 #idempotente
(
    chamados_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.chamados_custos")
)

# Conferência dos dados gravados
display(chamados_custos.orderBy("sk_chamado"))

### 2) Perguntas do Case

In [0]:
# Expõe a tabela Gold como visão temporária para as consultas SQL das análises seguintes
spark.table(f"{gold_schema}.chamados_custos").createOrReplaceTempView("fato_custos")

### Quantidade de problemas simples resolvidos em canais complexos e suas implicações

* Total e custo médio de referência: calcula o total de chamados resolvidos de baixa criticidade e o custo médio dos chamados de baixa criticidade atendidos pelo autosserviço.

* Quantidade e participação percentual: contabiliza os chamados resolvidos por cada canal humano e calcula sua porcentagem em relação ao total de chamados resolvidos de baixa criticidade.

* Custo total e custo excedente: soma os custos dos atendimentos humanos e estima o custo excedente em comparação ao cenário em que os mesmos chamados fossem atendidos pelo autosserviço.

* Filtros e organização: considera apenas chamados resolvidos, de baixa criticidade e atendidos pelo canal humano. Agrupa os resultados por canal e os ordena pelo custo excedente, do maior para o menor.

In [0]:
# Análise 1 — chamados simples (resolvidos, baixa criticidade) atendidos pelo canal humano:
# compara o custo real do atendimento humano com o custo estimado caso o autosserviço atendesse
display(
    spark.sql("""
        WITH total_chamados_simples AS (
            -- Total de chamados resolvidos de baixa criticidade (denominador da porcentagem)
            SELECT count(*) AS total
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.criticidade = 'baixa'
        ),
        custo_medio_canal_simples AS (
            -- Custo médio por chamado simples no autosserviço (referência da estimativa)
            SELECT round(sum(f.custo) / count(*), 2) AS custo_medio
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.tipo_canal = 'autosservico'
            AND f.criticidade = 'baixa'
        )
        -- Por canal humano: quantidade, participação %, custo total e custo excedente
        SELECT f.nome_canal AS Nome_canal, count(*) AS Qtd_Chamadas,
        round(100.0*count(*) / NULLIF((SELECT total FROM total_chamados_simples), 0), 2) AS Porcentagem,
        sum(f.custo) AS Custo_Total, 
        (sum(f.custo) - round(count(*) * (SELECT custo_medio FROM custo_medio_canal_simples), 2)) AS Custo_Excedente
        FROM fato_custos f
        WHERE f.resolvido = 'sim' AND f.tipo_canal = 'humano'
        AND f.criticidade = 'baixa'
        GROUP BY f.nome_canal
        ORDER BY Custo_Excedente DESC
    """)
)

### Quantidade de problemas complexos resolvidos em canais simples e suas implicações

* Total e custo médio de referência: calcula o total de chamados resolvidos de média e alta criticidade e o custo médio desses chamados quando atendidos pelo canal humano.

* Quantidade e participação percentual: contabiliza os chamados complexos resolvidos por cada canal de autosserviço e calcula sua porcentagem em relação ao total de chamados resolvidos de média e alta criticidade.

* Custo total e custo salvo: soma os custos dos atendimentos por autosserviço e estima a economia em comparação ao custo que esses mesmos chamados teriam se fossem atendidos pelo canal humano, utilizando o custo médio humano como referência.

* Filtros e organização: considera apenas chamados resolvidos, de média ou alta criticidade e atendidos pelo autosserviço. Agrupa os resultados por canal e os ordena pelo custo salvo, do maior para o menor.

Objetivo geral: identificar os canais de autosserviço que resolvem chamados complexos e estimar a economia potencial em comparação ao atendimento humano.

In [0]:
# Análise 2 — chamados complexos (resolvidos, média ou alta criticidade) atendidos pelo autosserviço:
# estima a economia (custo salvo) em relação ao custo médio do atendimento humano
display(
    spark.sql("""
        WITH total_chamados_complexas AS (
            -- Total de chamados resolvidos de média ou alta criticidade (denominador da porcentagem)
            SELECT count(*) AS total
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        ),
        custo_medio_canal_complexo AS (
            -- Custo médio por chamado complexo no canal humano (referência da estimativa)
            SELECT round(sum(f.custo) / count(*), 2) AS custo_medio
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.tipo_canal = 'humano'
            AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        )
        -- Por canal de autosserviço: quantidade, participação %, custo total e custo salvo
        SELECT f.nome_canal AS Nome_canal, count(*) AS Qtd_Chamadas,
        round(100.0*count(*) /  NULLIF((SELECT total FROM total_chamados_complexas), 0), 2) AS Porcentagem,
        sum(f.custo) AS Custo_Total, 
        (round(count(*) * (SELECT custo_medio FROM custo_medio_canal_complexo), 2) - sum(f.custo)) AS Custo_Salvo
        FROM fato_custos f
        WHERE f.resolvido = 'sim' AND f.tipo_canal = 'autosservico'
        AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        GROUP BY f.nome_canal
        ORDER BY Custo_Salvo DESC
    """)
)